# NUS ST5212 — Advanced Survival Analysis & Modern Survival ML

## Notebook 2: From Cox PH to survival forests, boosting, Coxnet, DeepSurv, IPCW evaluation, competing risks, recurrent events, frailty and flexible splines

This notebook is a **follow-up to the classical ST5212 notebook**. The first notebook focused on:

- censoring and risk sets;
- survival, hazard and cumulative hazard;
- Kaplan–Meier and Nelson–Aalen;
- log-rank testing;
- parametric survival models;
- Cox proportional hazards and partial likelihood;
- diagnostics and classical extensions.

This notebook asks:

> **What happens when the covariate–risk relationship is nonlinear, high-dimensional, clustered, recurrent, or involves competing event types?**

The goal is not to replace classical survival theory. It is to show that modern survival ML still rests on the same censoring-aware foundations.

### Main real-world case study

We continue with the **Rossi recidivism dataset**:

- 432 released individuals;
- follow-up up to one year;
- event = re-arrest;
- right censoring if no re-arrest is observed during follow-up.

This lets us compare classical Cox regression and modern survival models on the **same outcome and risk sets**.

### Additional mini-case studies

Some topics require a different event structure:

1. **BMT competing-risks data** — transplant-related mortality versus relapse;
2. **recurrent soreness-event data** — multiple events per individual;
3. the recurrent data are also used for a transparent shared-frailty demonstration.

### Learning objectives

You will implement and interpret:

1. Random Survival Forests;
2. Gradient Boosting Survival Analysis;
3. Coxnet Lasso / Elastic Net;
4. a DeepSurv-style neural Cox model;
5. Harrell versus Uno C-index;
6. IPCW;
7. time-dependent AUC;
8. time-dependent Brier score and IBS;
9. calibration curves at fixed horizons;
10. nested cross-validation;
11. competing-risk cumulative incidence and cause-specific prediction;
12. recurrent-event counting-process regression;
13. frailty/random-effects intuition;
14. flexible spline-based parametric survival models.

## 1. Why ordinary ML evaluation breaks under censoring

For subject $i$ we observe

$$
Y_i=\min(T_i,C_i),
$$

and

$$
\delta_i=I(T_i\le C_i),
$$

where:

- $T_i$ = true event time;
- $C_i$ = censoring time;
- $\delta_i=1$ = event observed.

If $\delta_i=0$, then the exact target $T_i$ is unknown. Therefore a normal regression loss such as

$$
(T_i-\hat T_i)^2
$$

cannot generally be computed.

Likewise, a survival label is time-dependent:

$$
Y_i(t)=I(T_i\le t).
$$

This is why survival ML needs censoring-aware:

- objectives;
- metrics;
- calibration;
- cross-validation.

## 2. Environment

Core libraries:

- `scikit-survival` for survival ML and censoring-aware metrics;
- `lifelines` for classical / recurrent / spline models;
- `scikit-learn` for model-selection infrastructure;
- `bokeh` for interactive visualization;
- `torch` for the neural Cox section.

PyTorch is not installed automatically because it is a large dependency. The DeepSurv section skips cleanly if it is unavailable.

In [ ]:
# Run once in a fresh environment.
%pip install -q "scikit-survival>=0.25,<0.29" "lifelines>=0.30,<0.31" "bokeh>=3.4" "scikit-learn>=1.5" pandas numpy scipy
# If needed for the DeepSurv section:
# %pip install -q torch

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from itertools import product
from typing import Dict, Mapping, Sequence, Tuple
import copy
import warnings

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import gammaln

from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from lifelines import KaplanMeierFitter, CoxTimeVaryingFitter, CRCSplineFitter
from lifelines.datasets import load_rossi, load_recur

from sksurv.datasets import load_bmt
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.metrics import (
    brier_score,
    concordance_index_censored,
    concordance_index_ipcw,
    cumulative_dynamic_auc,
    integrated_brier_score,
)
from sksurv.nonparametric import (
    CensoringDistributionEstimator,
    cumulative_incidence_competing_risks,
)
from sksurv.util import Surv

from bokeh.io import output_notebook, show
from bokeh.layouts import column
from bokeh.plotting import figure

output_notebook()
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Environment imported successfully.")

## 3. Reusable configuration and repository pattern

We avoid scattering constants and dataset translation logic throughout the notebook.

The repository converts the Rossi outcome into the structured-array representation expected by `scikit-survival`:

$$
(\delta_i,T_i).
$$

In [ ]:
@dataclass(frozen=True)
class Config:
    random_state: int = 42
    test_size: float = 0.25
    fast_mode: bool = True
    include_historical_race_indicator: bool = False
    calibration_bins: int = 5

CFG = Config()


class RossiRepository:
    """Load Rossi and construct a scikit-survival target."""

    BASE_FEATURES = ["fin", "age", "wexp", "mar", "paro", "prio"]

    def load(self, include_race: bool = False):
        df = load_rossi().copy()
        features = self.BASE_FEATURES + (["race"] if include_race else [])
        X = df[features].astype(float)
        y = Surv.from_arrays(
            event=df["arrest"].astype(bool).to_numpy(),
            time=df["week"].astype(float).to_numpy(),
            name_event="event",
            name_time="time",
        )
        return X, y, df


repo = RossiRepository()
X, y, rossi = repo.load(CFG.include_historical_race_indicator)

print("X shape:", X.shape)
print("Events:", int(y["event"].sum()))
print("Censored:", int((~y["event"]).sum()))
display(rossi.head())

### Modelling note

The original Rossi data contain historically coded demographic attributes. The race indicator is excluded from the **default predictive feature set**.

The statistical focus is on:

- censoring;
- event timing;
- risk ranking;
- survival-probability prediction;
- model assumptions;
- evaluation.

You can explicitly enable the field if your goal is to reproduce a historical specification.

In [ ]:
train_idx, test_idx = train_test_split(
    np.arange(len(X)),
    test_size=CFG.test_size,
    random_state=CFG.random_state,
    stratify=y["event"].astype(int),
)

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y[train_idx]
y_test = y[test_idx]

split_summary = pd.DataFrame({
    "split": ["train", "test"],
    "n": [len(y_train), len(y_test)],
    "events": [int(y_train["event"].sum()), int(y_test["event"].sum())],
    "event_rate": [y_train["event"].mean(), y_test["event"].mean()],
    "max_followup": [y_train["time"].max(), y_test["time"].max()],
})
display(split_summary.round(3))

## 4. A common prediction interface

Survival estimators often produce:

### Risk score

$$
r(x),
$$

which is enough for ranking metrics.

### Survival function

$$
\hat S(t\mid x),
$$

which is required for:

- Brier score;
- IBS;
- calibration;
- individual probability forecasts.

We create a common adapter for both.

In [ ]:
class SurvivalPredictionAdapter:
    @staticmethod
    def survival_matrix(model, X, times):
        fns = model.predict_survival_function(X)
        return np.row_stack([
            np.asarray([float(fn(t)) for t in times], dtype=float)
            for fn in fns
        ])

    @staticmethod
    def risk(model, X):
        return np.asarray(model.predict(X), dtype=float).reshape(-1)


def safe_eval_times(y_train, y_test, n_points=18):
    """Initial human-readable grid; IPCW support is enforced separately below."""
    event_times = y_train["time"][y_train["event"]]
    lo = max(float(np.quantile(event_times, 0.15)), 1.0)
    hi = min(
        float(np.quantile(event_times, 0.85)),
        float(y_train["time"].max()) - 1e-6,
        float(y_test["time"].max()) - 1e-6,
    )
    return np.linspace(lo, hi, n_points)


def inspect_censoring_support(y_train, y_test):
    """
    Inspect G_hat(t)=P(C>t) at every observed test-event time.

    cumulative_dynamic_auc() constructs IPCW weights for observed events in
    survival_test. Therefore a late test event can fail even when the requested
    AUC time grid itself is earlier.
    """
    censoring = CensoringDistributionEstimator().fit(y_train)
    test_event_times = np.sort(
        np.unique(y_test["time"][y_test["event"]].astype(float))
    )

    rows = []
    for t in test_event_times:
        try:
            g = float(censoring.predict_proba(np.asarray([t], dtype=float))[0])
        except ValueError:
            g = 0.0

        rows.append({
            "test_event_time": float(t),
            "G_hat": g,
            "IPCW_weight": np.inf if g <= 0.0 else 1.0 / g,
            "valid_for_IPCW": bool(g > 0.0),
        })

    return pd.DataFrame(rows)


def find_ipcw_safe_horizon(y_train, y_test=None, eps=1e-10):
    """
    Largest observed candidate time for which training censoring survival
    G_hat(t) remains strictly positive.

    IPCW positivity requirement:
        G_hat(t) > 0.
    """
    censoring = CensoringDistributionEstimator().fit(y_train)

    candidate_times = y_train["time"].astype(float)
    if y_test is not None:
        candidate_times = np.concatenate([
            candidate_times,
            y_test["time"].astype(float),
        ])

    candidate_times = np.sort(np.unique(candidate_times))
    valid_times = []

    for t in candidate_times:
        try:
            g = float(censoring.predict_proba(np.asarray([t], dtype=float))[0])
        except ValueError:
            break

        if np.isfinite(g) and g > eps:
            valid_times.append(float(t))
        else:
            break

    if not valid_times:
        raise ValueError(
            "No IPCW-valid horizon exists: training censoring survival "
            "reaches zero before a usable evaluation time."
        )

    return max(valid_times)


def administrative_truncate(y, tau):
    """
    Administratively censor follow-up at tau while retaining every subject.

    If an observed event occurs after tau, it becomes censored at tau for the
    horizon-limited evaluation problem.
    """
    times = y["time"].astype(float)
    events = y["event"].astype(bool)

    truncated_time = np.minimum(times, tau)
    truncated_event = events & (times <= tau)

    return Surv.from_arrays(
        event=truncated_event,
        time=truncated_time,
        name_event="event",
        name_time="time",
    )


def make_ipcw_safe_evaluation(y_train, y_test, requested_times, eps=1e-10):
    """
    Build a censoring-support-safe evaluation problem.

    Returns
    -------
    tau : float
        Administrative horizon strictly inside positive G_hat support.
    y_test_ipcw : structured array
        Test outcome administratively censored at tau.
    metric_times : ndarray
        Requested metric grid restricted to times < tau.
    """
    boundary = find_ipcw_safe_horizon(
        y_train=y_train,
        y_test=y_test,
        eps=eps,
    )

    # Stay infinitesimally inside the last supported time so that an event
    # exactly on a problematic boundary cannot acquire an undefined weight.
    tau = float(np.nextafter(boundary, -np.inf))

    y_test_ipcw = administrative_truncate(y_test, tau=tau)

    metric_times = np.asarray(requested_times, dtype=float)
    metric_times = metric_times[
        np.isfinite(metric_times)
        & (metric_times > 0.0)
        & (metric_times < tau)
    ]
    metric_times = np.unique(metric_times)

    if metric_times.size < 2:
        train_event_times = y_train["time"][y_train["event"]].astype(float)
        lo = max(float(np.quantile(train_event_times, 0.10)), 1.0)
        hi = min(float(np.quantile(train_event_times, 0.80)), tau)

        if not hi > lo:
            raise ValueError(
                f"Insufficient IPCW-supported interval: lo={lo}, hi={hi}."
            )

        metric_times = np.linspace(lo, hi, 15, endpoint=False)
        metric_times = metric_times[metric_times < tau]

    return tau, y_test_ipcw, metric_times


EVAL_TIMES = safe_eval_times(y_train, y_test)

IPCW_TAU, y_test_ipcw, METRIC_TIMES = make_ipcw_safe_evaluation(
    y_train=y_train,
    y_test=y_test,
    requested_times=EVAL_TIMES,
)

print("Initial evaluation range:", EVAL_TIMES[0], "to", EVAL_TIMES[-1])
print("IPCW-safe horizon:", IPCW_TAU)
print("IPCW metric range:", METRIC_TIMES[0], "to", METRIC_TIMES[-1])


# Part I — Cox PH as the reference model

Cox PH assumes

$$
h(t\mid x)=h_0(t)\exp(x^\top\beta).
$$

Its main functional restriction is

$$
f(x)=x^\top\beta.
$$

Modern survival ML frequently keeps the censoring-aware survival machinery while replacing this simple linear $f(x)$.

In [ ]:
cox = make_pipeline(
    StandardScaler(),
    CoxPHSurvivalAnalysis(alpha=1e-4, ties="efron"),
)
cox.fit(X_train, y_train)

cox_risk = cox.predict(X_test)
cox_c = concordance_index_censored(
    y_test["event"], y_test["time"], cox_risk
)[0]

print(f"Baseline Cox test Harrell C-index: {cox_c:.4f}")

# Part II — Random Survival Forests

## 5. How RSF works

An ordinary decision tree uses classification or regression impurity.

A survival tree must respect censoring.

A common split criterion asks whether two candidate child nodes have different survival experience, often using a log-rank-style statistic:

$$
Z_{\text{split}}
\approx
\frac{O_L-E_L}
{\sqrt{\operatorname{Var}(O_L-E_L)}}.
$$

Inside a terminal node, a survival or cumulative-hazard estimator is computed from the observations that reach that node.

Across $B$ trees,

$$
\hat S_{\text{RSF}}(t\mid x)
=
\frac1B
\sum_{b=1}^{B}
\hat S_b(t\mid x).
$$

RSF can automatically learn:

- thresholds;
- nonlinear effects;
- interactions;
- non-additive risk structures.

In [ ]:
rsf = RandomSurvivalForest(
    n_estimators=150 if CFG.fast_mode else 400,
    min_samples_split=10,
    min_samples_leaf=8,
    max_features="sqrt",
    n_jobs=-1,
    random_state=CFG.random_state,
)
rsf.fit(X_train, y_train)

rsf_risk = rsf.predict(X_test)
rsf_c = concordance_index_censored(
    y_test["event"], y_test["time"], rsf_risk
)[0]

print(f"RSF test Harrell C-index: {rsf_c:.4f}")

## 6. RSF hyperparameter intuition

### `n_estimators`

More trees mostly reduce Monte-Carlo variance.

### `min_samples_leaf`

Small leaves:

- lower bias;
- higher variance;
- noisier terminal-node survival estimates.

Large leaves:

- smoother estimates;
- lower variance;
- greater underfitting risk.

### `max_features`

Restricting the feature subset decorrelates trees.

Averaging is most effective when base learners are:

1. individually useful;
2. not perfectly correlated.

In [ ]:
def evaluate_rsf_grid(X_train, y_train, X_test, y_test, fast_mode=True):
    n_estimators = [80, 160] if fast_mode else [100, 300, 600]
    leaf_sizes = [4, 10, 20]
    max_features_values = ["sqrt", 0.8]

    rows = []
    for n_est, leaf, max_feat in product(
        n_estimators, leaf_sizes, max_features_values
    ):
        model = RandomSurvivalForest(
            n_estimators=n_est,
            min_samples_split=max(6, 2 * leaf),
            min_samples_leaf=leaf,
            max_features=max_feat,
            n_jobs=-1,
            random_state=CFG.random_state,
        )
        model.fit(X_train, y_train)
        risk = model.predict(X_test)
        c = concordance_index_censored(
            y_test["event"], y_test["time"], risk
        )[0]
        rows.append({
            "n_estimators": n_est,
            "min_samples_leaf": leaf,
            "max_features": str(max_feat),
            "c_index": c,
        })

    return pd.DataFrame(rows).sort_values("c_index", ascending=False)


rsf_grid_df = evaluate_rsf_grid(
    X_train, y_train, X_test, y_test, CFG.fast_mode
)
display(rsf_grid_df.round(4))

In [ ]:
p = figure(
    width=820,
    height=420,
    title="RSF sensitivity: leaf size vs test C-index",
    x_axis_label="min_samples_leaf",
    y_axis_label="Harrell C-index",
)

for (n_est, max_feat), grp in rsf_grid_df.groupby(
    ["n_estimators", "max_features"]
):
    grp = grp.sort_values("min_samples_leaf")
    label = f"trees={n_est}, max_features={max_feat}"
    p.line(
        grp["min_samples_leaf"],
        grp["c_index"],
        line_width=2,
        legend_label=label,
    )
    p.scatter(
        grp["min_samples_leaf"],
        grp["c_index"],
        size=8,
        legend_label=label,
    )

p.legend.location = "top_right"
p.legend.click_policy = "hide"
show(p)

### Inference from the RSF experiment

The aim is not to memorize an optimal leaf size.

The point is to observe the bias–variance mechanism:

$$
\text{leaf size}\downarrow
\Rightarrow
\text{local flexibility}\uparrow
\Rightarrow
\text{variance}\uparrow.
$$

Because Rossi is small, performance differences can be dominated by sampling variability. Nested CV later gives a more defensible estimate.

# Part III — Gradient Boosting Survival Analysis

## 7. Functional gradient descent with Cox loss

Gradient boosting constructs

$$
f_M(x)
=
\sum_{m=1}^{M}\nu g_m(x),
$$

where $g_m$ is typically a shallow tree.

For Cox-style boosting,

$$
h(t\mid x)=h_0(t)\exp(f(x)).
$$

The loss is negative partial log-likelihood:

$$
\mathcal L(f)
=
-
\sum_{i:\delta_i=1}
\left[
f(x_i)
-
\log\sum_{j\in R_i}e^{f(x_j)}
\right].
$$

At each iteration, a tree approximates the **negative functional gradient**.

So this model can be understood as:

> Cox's risk-set likelihood + a nonlinear additive tree representation for log-risk.

In [ ]:
gb = GradientBoostingSurvivalAnalysis(
    loss="coxph",
    learning_rate=0.04,
    n_estimators=140 if CFG.fast_mode else 300,
    max_depth=2,
    min_samples_leaf=8,
    subsample=0.8,
    random_state=CFG.random_state,
)
gb.fit(X_train, y_train)

gb_risk = gb.predict(X_test)
gb_c = concordance_index_censored(
    y_test["event"], y_test["time"], gb_risk
)[0]

print(f"Gradient Boosting Survival test C-index: {gb_c:.4f}")

## 8. Learning rate and number of estimators

The update is

$$
f_m(x)
=
f_{m-1}(x)+\nu g_m(x).
$$

Usually:

$$
\nu\downarrow
\Longrightarrow
M\uparrow.
$$

A small learning rate makes each tree's contribution conservative.

`subsample < 1` introduces stochastic boosting and acts as regularization.

In [ ]:
def evaluate_gb_grid(X_train, y_train, X_test, y_test, fast_mode=True):
    learning_rates = [0.03, 0.08]
    n_estimators_values = [100, 220] if fast_mode else [100, 250, 500]
    depths = [1, 2, 3]
    subsamples = [0.7, 1.0]

    rows = []
    for lr, n_est, depth, subsample in product(
        learning_rates, n_estimators_values, depths, subsamples
    ):
        model = GradientBoostingSurvivalAnalysis(
            loss="coxph",
            learning_rate=lr,
            n_estimators=n_est,
            max_depth=depth,
            min_samples_leaf=8,
            subsample=subsample,
            random_state=CFG.random_state,
        )
        model.fit(X_train, y_train)
        risk = model.predict(X_test)
        c = concordance_index_censored(
            y_test["event"], y_test["time"], risk
        )[0]
        rows.append({
            "learning_rate": lr,
            "n_estimators": n_est,
            "max_depth": depth,
            "subsample": subsample,
            "c_index": c,
        })

    return pd.DataFrame(rows).sort_values("c_index", ascending=False)


gb_grid_df = evaluate_gb_grid(
    X_train, y_train, X_test, y_test, CFG.fast_mode
)
display(gb_grid_df.head(15).round(4))

In [ ]:
p = figure(
    width=820,
    height=420,
    title="Gradient boosting: complexity vs discrimination",
    x_axis_label="n_estimators",
    y_axis_label="Harrell C-index",
)

for (lr, depth, subsample), grp in gb_grid_df.groupby(
    ["learning_rate", "max_depth", "subsample"]
):
    grp = grp.sort_values("n_estimators")
    label = f"lr={lr}, depth={depth}, subsample={subsample}"
    p.line(
        grp["n_estimators"],
        grp["c_index"],
        line_width=2,
        legend_label=label,
    )
    p.scatter(
        grp["n_estimators"],
        grp["c_index"],
        size=7,
        legend_label=label,
    )

p.legend.location = "top_right"
p.legend.click_policy = "hide"
show(p)

# Part IV — Coxnet: Lasso and Elastic Net

## 9. Penalized partial likelihood

Coxnet solves

$$
\arg\max_\beta
\left[
\ell_{\text{partial}}(\beta)
-
\alpha
\left(
r\sum_j|\beta_j|
+
\frac{1-r}{2}\sum_j\beta_j^2
\right)
\right].
$$

- $r=1$ gives Lasso;
- $r\approx0$ behaves more like Ridge;
- intermediate $r$ gives Elastic Net.

Lasso can perform feature selection.

Elastic Net is often more stable when features are correlated.

In [ ]:
scaler_path = StandardScaler()
X_train_scaled = scaler_path.fit_transform(X_train)

coxnet_path = CoxnetSurvivalAnalysis(
    l1_ratio=0.9,
    alpha_min_ratio=0.01,
    n_alphas=60,
)
coxnet_path.fit(X_train_scaled, y_train)

coef_path = pd.DataFrame(
    coxnet_path.coef_,
    index=X_train.columns,
    columns=coxnet_path.alphas_,
)

print("Alphas on path:", len(coxnet_path.alphas_))
display(coef_path.iloc[:, ::10].round(4))

In [ ]:
p = figure(
    width=850,
    height=450,
    x_axis_type="log",
    title="Coxnet coefficient path",
    x_axis_label="alpha (log scale)",
    y_axis_label="coefficient",
)

for feature in coef_path.index:
    p.line(
        coef_path.columns.to_numpy(dtype=float),
        coef_path.loc[feature].to_numpy(dtype=float),
        line_width=2,
        legend_label=feature,
    )

p.legend.location = "top_right"
p.legend.click_policy = "hide"
show(p)

### Reading the path

Large $\alpha$ means strong regularization.

As $\alpha$ falls:

- coefficients move away from zero;
- more predictors can enter;
- variance increases.

A coefficient path is useful because it shows **stability across regularization strength**, not just one selected fit.

In [ ]:
def stratified_survival_splits(y_struct, n_splits=4, seed=42):
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )
    return list(
        skf.split(
            np.zeros(len(y_struct)),
            y_struct["event"].astype(int),
        )
    )


candidate_alphas = np.unique(
    np.quantile(
        coxnet_path.alphas_,
        np.linspace(0.05, 0.95, 10 if CFG.fast_mode else 18),
    )
)

cv_splits = stratified_survival_splits(
    y_train,
    n_splits=3 if CFG.fast_mode else 5,
    seed=CFG.random_state,
)

alpha_rows = []

for alpha in candidate_alphas:
    fold_scores = []
    for tr, va in cv_splits:
        model = make_pipeline(
            StandardScaler(),
            CoxnetSurvivalAnalysis(
                l1_ratio=0.9,
                alphas=[float(alpha)],
                fit_baseline_model=True,
            ),
        )
        model.fit(X_train.iloc[tr], y_train[tr])
        fold_scores.append(
            model.score(X_train.iloc[va], y_train[va])
        )

    alpha_rows.append({
        "alpha": float(alpha),
        "cv_c_index_mean": np.mean(fold_scores),
        "cv_c_index_sd": np.std(fold_scores, ddof=1),
    })

coxnet_cv = pd.DataFrame(alpha_rows).sort_values(
    "cv_c_index_mean",
    ascending=False,
)
display(coxnet_cv.round(4))

best_alpha = float(coxnet_cv.iloc[0]["alpha"])
print("Selected alpha:", best_alpha)

In [ ]:
coxnet = make_pipeline(
    StandardScaler(),
    CoxnetSurvivalAnalysis(
        l1_ratio=0.9,
        alphas=[best_alpha],
        fit_baseline_model=True,
    ),
)
coxnet.fit(X_train, y_train)

coxnet_risk = coxnet.predict(X_test)
coxnet_c = concordance_index_censored(
    y_test["event"], y_test["time"], coxnet_risk
)[0]

print(f"Coxnet Elastic-Net test C-index: {coxnet_c:.4f}")

# Part V — DeepSurv / neural Cox

## 10. Neuralizing the Cox predictor

Cox PH uses

$$
f(x)=x^\top\beta.
$$

A DeepSurv-style model replaces it by

$$
f_\theta(x)=\operatorname{NN}_\theta(x).
$$

Then

$$
h(t\mid x)
=
h_0(t)\exp(f_\theta(x)).
$$

The model still trains with a Cox partial-likelihood objective.

Because Rossi event times are recorded in weeks, tied events are common. The implementation below therefore includes an **Efron tied-event correction**.

In [ ]:
try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
    torch.manual_seed(CFG.random_state)
except Exception:
    HAS_TORCH = False

print("PyTorch available:", HAS_TORCH)

if not HAS_TORCH:
    print("Install with: %pip install torch")

In [ ]:
if HAS_TORCH:
    class DeepSurvNet(nn.Module):
        def __init__(self, n_features, hidden=(32, 16), dropout=0.15):
            super().__init__()
            layers = []
            in_dim = n_features

            for width in hidden:
                layers.extend([
                    nn.Linear(in_dim, width),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                ])
                in_dim = width

            layers.append(nn.Linear(in_dim, 1))
            self.network = nn.Sequential(*layers)

        def forward(self, x):
            return self.network(x).squeeze(-1)


    def efron_negative_partial_log_likelihood(
        log_risk,
        durations,
        events,
    ):
        events = events.bool()
        unique_event_times = torch.unique(durations[events])
        pll = torch.tensor(0.0, device=log_risk.device)
        n_events = torch.clamp(events.sum(), min=1)

        exp_risk = torch.exp(
            torch.clamp(log_risk, -30.0, 30.0)
        )

        for t in unique_event_times:
            death_mask = (durations == t) & events
            risk_mask = durations >= t
            d = int(death_mask.sum().item())

            if d == 0:
                continue

            pll = pll + log_risk[death_mask].sum()
            risk_sum = exp_risk[risk_mask].sum()
            death_sum = exp_risk[death_mask].sum()

            for l in range(d):
                denom = risk_sum - (l / d) * death_sum
                pll = pll - torch.log(
                    torch.clamp(denom, min=1e-12)
                )

        return -pll / n_events


    class DeepSurvEstimator:
        """Small pedagogical sklearn-like DeepSurv wrapper."""

        def __init__(
            self,
            hidden=(32, 16),
            dropout=0.15,
            lr=1e-3,
            weight_decay=1e-4,
            epochs=350,
            patience=35,
            random_state=42,
        ):
            self.hidden = hidden
            self.dropout = dropout
            self.lr = lr
            self.weight_decay = weight_decay
            self.epochs = epochs
            self.patience = patience
            self.random_state = random_state

        def fit(self, X, y):
            torch.manual_seed(self.random_state)

            self.scaler_ = StandardScaler()
            Xs = self.scaler_.fit_transform(
                np.asarray(X, dtype=float)
            )

            event = y["event"].astype(bool)
            time = y["time"].astype(float)

            tr, va = train_test_split(
                np.arange(len(Xs)),
                test_size=0.18,
                random_state=self.random_state,
                stratify=event.astype(int),
            )

            Xtr = torch.tensor(Xs[tr], dtype=torch.float32)
            Ttr = torch.tensor(time[tr], dtype=torch.float32)
            Etr = torch.tensor(event[tr], dtype=torch.bool)

            Xva = torch.tensor(Xs[va], dtype=torch.float32)
            Tva = torch.tensor(time[va], dtype=torch.float32)
            Eva = torch.tensor(event[va], dtype=torch.bool)

            self.net_ = DeepSurvNet(
                n_features=Xs.shape[1],
                hidden=self.hidden,
                dropout=self.dropout,
            )

            optimizer = torch.optim.Adam(
                self.net_.parameters(),
                lr=self.lr,
                weight_decay=self.weight_decay,
            )

            best_state = None
            best_val = np.inf
            bad_epochs = 0
            self.history_ = []

            for epoch in range(self.epochs):
                self.net_.train()
                optimizer.zero_grad()

                loss = efron_negative_partial_log_likelihood(
                    self.net_(Xtr),
                    Ttr,
                    Etr,
                )
                loss.backward()
                optimizer.step()

                self.net_.eval()
                with torch.no_grad():
                    val_loss = efron_negative_partial_log_likelihood(
                        self.net_(Xva),
                        Tva,
                        Eva,
                    ).item()

                self.history_.append(
                    (epoch, float(loss.item()), val_loss)
                )

                if val_loss < best_val - 1e-5:
                    best_val = val_loss
                    best_state = copy.deepcopy(
                        self.net_.state_dict()
                    )
                    bad_epochs = 0
                else:
                    bad_epochs += 1

                if bad_epochs >= self.patience:
                    break

            self.net_.load_state_dict(best_state)
            self._fit_breslow_baseline(X, y)
            return self

        def predict(self, X):
            Xs = self.scaler_.transform(
                np.asarray(X, dtype=float)
            )
            self.net_.eval()

            with torch.no_grad():
                return self.net_(
                    torch.tensor(Xs, dtype=torch.float32)
                ).numpy()

        def _fit_breslow_baseline(self, X, y):
            eta = self.predict(X)
            exp_eta = np.exp(np.clip(eta, -30, 30))
            times = y["time"].astype(float)
            events = y["event"].astype(bool)
            event_times = np.sort(
                np.unique(times[events])
            )

            increments = []
            for t in event_times:
                d = np.sum(events & (times == t))
                denom = exp_eta[times >= t].sum()
                increments.append(d / denom)

            self.baseline_times_ = event_times
            self.baseline_cumhaz_ = np.cumsum(increments)

        def predict_survival_function(self, X):
            eta = self.predict(X)
            fns = []

            for score in eta:
                surv = np.exp(
                    -self.baseline_cumhaz_
                    * np.exp(np.clip(score, -30, 30))
                )
                fns.append(
                    StepFunction(
                        self.baseline_times_,
                        surv,
                        a=1.0,
                        b=0.0,
                    )
                )

            return fns


    deep = DeepSurvEstimator(
        epochs=220 if CFG.fast_mode else 500,
        patience=30,
        random_state=CFG.random_state,
    )
    deep.fit(X_train, y_train)

    deep_risk = deep.predict(X_test)
    deep_c = concordance_index_censored(
        y_test["event"],
        y_test["time"],
        deep_risk,
    )[0]

    print(f"DeepSurv-style test C-index: {deep_c:.4f}")
else:
    deep = None

In [ ]:
if HAS_TORCH:
    hist = pd.DataFrame(
        deep.history_,
        columns=["epoch", "train_loss", "validation_loss"],
    )

    p = figure(
        width=820,
        height=400,
        title="DeepSurv training history",
        x_axis_label="epoch",
        y_axis_label="negative partial log-likelihood",
    )
    p.line(
        hist["epoch"],
        hist["train_loss"],
        line_width=2,
        legend_label="train",
    )
    p.line(
        hist["epoch"],
        hist["validation_loss"],
        line_width=2,
        legend_label="validation",
    )
    p.legend.location = "top_right"
    show(p)

### Why DeepSurv may not win

Neural models have much greater function capacity:

$$
x^\top\beta
\longrightarrow
f_\theta(x).
$$

But Rossi contains only 432 subjects.

Greater function capacity also means greater estimation variance.

This is an important survival-ML lesson:

> **Model expressiveness must be justified by event count and sample size.**

# Part VI — Censoring-aware evaluation

## 11. Harrell C-index

For comparable subjects $i,j$, if

$$
T_i<T_j,
$$

we hope the model gives

$$
r_i>r_j.
$$

The C-index estimates ranking correctness.

However, Harrell's estimator can become biased when censoring is substantial.

## 12. Uno C-index and IPCW

Let

$$
G(t)=P(C>t)
$$

be the censoring survival function.

IPCW uses weights related to

$$
\frac{1}{\hat G(t)}.
$$

If few observations remain observable late in follow-up, each observed subject carries more information about the original population.

This weighting principle powers:

- Uno C-index;
- time-dependent Brier score;
- time-dependent ROC/AUC;
- many survival-learning procedures.

### IPCW positivity and administrative truncation

The censoring survival function is

$$
G(t)=P(C>t).
$$

IPCW metrics use factors proportional to

$$
\frac{1}{\hat G(t)}.
$$

Therefore they require the **positivity condition**

$$
\hat G(t)>0
$$

through the evaluation horizon. A late observed test event can violate this even if the requested AUC/Brier time grid itself is earlier, because `cumulative_dynamic_auc()` first computes IPCW weights for observed test events.

The robust fix is **administrative truncation** at a supported horizon $\tau$:

$$
T_i^*=\min(T_i,\tau),
$$

$$
\delta_i^*=\delta_i I(T_i\le\tau).
$$

This keeps every test subject but changes the estimand to prediction performance through $\tau$. We do **not** add an arbitrary epsilon to a zero censoring probability and we do **not** delete long-follow-up subjects.

In [ ]:
# Diagnose the exact positivity issue that can break IPCW metrics.
ipcw_support_diagnostic = inspect_censoring_support(
    y_train,
    y_test,
)

display(ipcw_support_diagnostic.tail(15).round(6))

invalid_ipcw_events = ipcw_support_diagnostic.loc[
    ~ipcw_support_diagnostic["valid_for_IPCW"]
]

print("Late test events with undefined IPCW weights:", len(invalid_ipcw_events))
print(f"Administrative IPCW evaluation horizon tau = {IPCW_TAU:.6f}")
print(
    "Original test events:",
    int(y_test["event"].sum()),
    "| events observed by tau:",
    int(y_test_ipcw["event"].sum()),
)

# Visualize how inverse censoring weights behave only on supported times.
censoring_estimator = CensoringDistributionEstimator().fit(y_train)

ipcw_grid = np.linspace(
    METRIC_TIMES[0],
    METRIC_TIMES[-1],
    12,
)

G_hat = censoring_estimator.predict_proba(ipcw_grid)

ipcw_demo = pd.DataFrame({
    "time": ipcw_grid,
    "G_hat_censoring_survival": G_hat,
    "inverse_weight_1_over_G": 1.0 / G_hat,
})

display(ipcw_demo.round(4))


In [ ]:
p = figure(
    width=800,
    height=400,
    title="Why IPCW weights grow later in follow-up",
    x_axis_label="time",
    y_axis_label="1 / G_hat(t)",
)

p.line(
    ipcw_demo["time"],
    ipcw_demo["inverse_weight_1_over_G"],
    line_width=3,
)
p.scatter(
    ipcw_demo["time"],
    ipcw_demo["inverse_weight_1_over_G"],
    size=8,
)

show(p)

## 13. Time-dependent Brier score

At horizon $t$, a model predicts

$$
\hat S(t\mid x_i).
$$

Ignoring censoring temporarily,

$$
BS(t)
=
\frac1n
\sum_i
\left[
I(T_i>t)-\hat S(t\mid x_i)
\right]^2.
$$

IPCW modifies this calculation when survival status at $t$ is not directly observable.

Lower Brier score is better.

### Integrated Brier Score

$$
IBS
=
\frac1{t_2-t_1}
\int_{t_1}^{t_2}
BS(t)\,dt.
$$

A useful distinction:

- C-index = ranking/discrimination;
- IBS = survival-probability accuracy.

In [ ]:
models = {
    "Cox PH": cox,
    "Coxnet EN": coxnet,
    "Random Survival Forest": rsf,
    "Gradient Boosting": gb,
}

if HAS_TORCH:
    models["DeepSurv"] = deep


def evaluate_models(
    models,
    X_test,
    y_train,
    y_test,
    requested_times,
):
    """
    Evaluate survival models without violating IPCW positivity.

    Harrell C uses the full test follow-up.

    Uno C, cumulative/dynamic AUC, Brier score and IBS use an
    administratively truncated test outcome restricted to the region where
    the training censoring survival estimate G_hat(t) is strictly positive.
    """
    ipcw_tau, y_test_for_ipcw, eval_times = make_ipcw_safe_evaluation(
        y_train=y_train,
        y_test=y_test,
        requested_times=requested_times,
    )

    rows = []
    brier_curves = {}
    auc_curves = {}

    for name, model in models.items():
        risk = SurvivalPredictionAdapter.risk(
            model,
            X_test,
        )

        survival = SurvivalPredictionAdapter.survival_matrix(
            model,
            X_test,
            eval_times,
        )

        # Full-follow-up ranking metric: does not need inverse censoring weights.
        harrell_c = concordance_index_censored(
            y_test["event"],
            y_test["time"],
            risk,
        )[0]

        # IPCW-dependent metrics: evaluate only inside censoring support.
        uno_c = concordance_index_ipcw(
            survival_train=y_train,
            survival_test=y_test_for_ipcw,
            estimate=risk,
            tau=ipcw_tau,
        )[0]

        bs_times, bs_values = brier_score(
            survival_train=y_train,
            survival_test=y_test_for_ipcw,
            estimate=survival,
            times=eval_times,
        )

        ibs = integrated_brier_score(
            survival_train=y_train,
            survival_test=y_test_for_ipcw,
            estimate=survival,
            times=eval_times,
        )

        auc_values, mean_auc = cumulative_dynamic_auc(
            survival_train=y_train,
            survival_test=y_test_for_ipcw,
            estimate=risk,
            times=eval_times,
        )

        rows.append({
            "model": name,
            "harrell_c_full_followup": harrell_c,
            "uno_c_ipcw": uno_c,
            "mean_dynamic_auc": mean_auc,
            "integrated_brier_score": ibs,
            "ipcw_tau": ipcw_tau,
        })

        brier_curves[name] = (
            bs_times,
            bs_values,
        )
        auc_curves[name] = (
            eval_times.copy(),
            auc_values,
        )

    result = (
        pd.DataFrame(rows)
        .sort_values(
            ["integrated_brier_score", "uno_c_ipcw"],
            ascending=[True, False],
        )
        .reset_index(drop=True)
    )

    return (
        result,
        brier_curves,
        auc_curves,
        eval_times,
        ipcw_tau,
        y_test_for_ipcw,
    )


(
    benchmark_df,
    brier_curves,
    auc_curves,
    METRIC_TIMES,
    IPCW_TAU,
    y_test_ipcw,
) = evaluate_models(
    models=models,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    requested_times=EVAL_TIMES,
)

display(benchmark_df.round(4))

print(
    f"IPCW-dependent metrics evaluated only through t={IPCW_TAU:.4f}. "
    "Harrell C uses the full test follow-up."
)


In [ ]:
p_auc = figure(
    width=830,
    height=420,
    title="Cumulative / dynamic AUC across time",
    x_axis_label="time",
    y_axis_label="AUC(t)",
    y_range=(0.4, 1.0),
)

for name, (times, aucs) in auc_curves.items():
    p_auc.line(
        times,
        aucs,
        line_width=2,
        legend_label=name,
    )

p_auc.legend.location = "top_right"
p_auc.legend.click_policy = "hide"
show(p_auc)

In [ ]:
p_bs = figure(
    width=830,
    height=420,
    title="IPCW time-dependent Brier score",
    x_axis_label="time",
    y_axis_label="Brier score (lower is better)",
)

for name, (times, scores) in brier_curves.items():
    p_bs.line(
        times,
        scores,
        line_width=2,
        legend_label=name,
    )

p_bs.legend.location = "top_right"
p_bs.legend.click_policy = "hide"
show(p_bs)

### How to interpret the benchmark

Do not ask only:

> Which model has the largest C-index?

Also ask:

1. Which model has the lowest IBS?
2. Is AUC stable across time?
3. Does a model rank well but produce poor probabilities?
4. Is a tiny improvement worth much greater complexity?
5. Is performance likely stable under resampling?

# Part VII — Fixed-horizon calibration

## 14. Ranking is not calibration

A model can rank subjects correctly while giving terrible absolute probabilities.

At horizon $t^\*$ we compare

$$
\hat S(t^\*\mid x)
$$

with observed survival.

Because some observations are censored before $t^\*$, raw proportions are inappropriate.

Within bins of similar predicted probability, we estimate observed survival using Kaplan–Meier.

In [ ]:
class SurvivalCalibration:
    @staticmethod
    def at_horizon(
        model,
        X,
        y,
        horizon,
        n_bins=5,
    ):
        fns = model.predict_survival_function(X)
        pred = np.asarray(
            [float(fn(horizon)) for fn in fns]
        )

        frame = pd.DataFrame({
            "pred_survival": pred,
            "time": y["time"],
            "event": y["event"].astype(int),
        })

        q = min(
            n_bins,
            max(2, frame["pred_survival"].nunique()),
        )

        frame["bin"] = pd.qcut(
            frame["pred_survival"],
            q=q,
            duplicates="drop",
        )

        rows = []

        for interval, grp in frame.groupby(
            "bin",
            observed=True,
        ):
            km = KaplanMeierFitter().fit(
                durations=grp["time"],
                event_observed=grp["event"],
            )

            rows.append({
                "bin": str(interval),
                "n": len(grp),
                "mean_predicted_survival": (
                    grp["pred_survival"].mean()
                ),
                "km_observed_survival": float(
                    km.predict(horizon)
                ),
            })

        return pd.DataFrame(rows)


CAL_HORIZON = 26.0

calibration_tables = {
    name: SurvivalCalibration.at_horizon(
        model,
        X_test,
        y_test,
        horizon=CAL_HORIZON,
        n_bins=CFG.calibration_bins,
    )
    for name, model in models.items()
}

display(calibration_tables["Cox PH"].round(4))

In [ ]:
p = figure(
    width=700,
    height=520,
    title=f"Calibration at week {CAL_HORIZON:.0f}",
    x_axis_label="Mean predicted survival probability",
    y_axis_label="Kaplan–Meier observed survival",
    x_range=(0.0, 1.0),
    y_range=(0.0, 1.0),
)

p.line(
    [0, 1],
    [0, 1],
    line_dash="dashed",
    line_width=2,
    legend_label="perfect calibration",
)

for name, table in calibration_tables.items():
    p.line(
        table["mean_predicted_survival"],
        table["km_observed_survival"],
        line_width=2,
        legend_label=name,
    )
    p.scatter(
        table["mean_predicted_survival"],
        table["km_observed_survival"],
        size=8,
        legend_label=name,
    )

p.legend.location = "bottom_right"
p.legend.click_policy = "hide"
show(p)

### Calibration caveat

The Rossi test set is small.

Therefore calibration bins are noisy.

For production evaluation, prefer:

- larger held-out sets;
- bootstrap confidence intervals;
- multiple horizons;
- explicit recalibration if needed.

# Part VIII — Representative survival curves

A survival estimator is most informative when it predicts a **function**, not merely a risk score.

We create three covariate profiles from training-set quartiles and compare predicted survival trajectories.

In [ ]:
profiles = pd.DataFrame({
    col: [
        X_train[col].quantile(0.25),
        X_train[col].median(),
        X_train[col].quantile(0.75),
    ]
    for col in X_train.columns
}, index=["Q25 profile", "Median profile", "Q75 profile"])

binary_cols = [
    c for c in X_train.columns
    if set(X_train[c].dropna().unique()).issubset(
        {0.0, 1.0}
    )
]

for c in binary_cols:
    profiles[c] = profiles[c].round().clip(0, 1)

display(profiles)

In [ ]:
curve_times = np.linspace(1, 51, 80)

plots = []

for model_name, model in models.items():
    surv = SurvivalPredictionAdapter.survival_matrix(
        model,
        profiles,
        curve_times,
    )

    p = figure(
        width=820,
        height=350,
        title=f"{model_name}: representative survival curves",
        x_axis_label="week",
        y_axis_label="P(no re-arrest beyond t)",
        y_range=(0.0, 1.0),
    )

    for i, profile_name in enumerate(profiles.index):
        p.line(
            curve_times,
            surv[i],
            line_width=2,
            legend_label=profile_name,
        )

    p.legend.location = "bottom_left"
    plots.append(p)

show(column(*plots))

# Part IX — Nested cross-validation

## 15. Why nested CV?

If the same validation information is used both to:

1. choose hyperparameters;
2. report performance,

the result is optimistic.

Nested CV separates:

### Inner loop

Choose hyperparameters.

### Outer loop

Estimate generalization.

The outer score evaluates the **whole tuning procedure**.

In [ ]:
@dataclass
class NestedModelSpec:
    name: str
    estimator: object
    param_grid: Mapping[str, Sequence]


def build_nested_specs(fast_mode=True):
    return [
        NestedModelSpec(
            name="RSF",
            estimator=RandomSurvivalForest(
                n_jobs=-1,
                random_state=CFG.random_state,
            ),
            param_grid={
                "n_estimators": (
                    [100, 220]
                    if fast_mode
                    else [100, 300, 600]
                ),
                "min_samples_leaf": [5, 12],
                "max_features": ["sqrt", 0.8],
            },
        ),
        NestedModelSpec(
            name="GradientBoosting",
            estimator=GradientBoostingSurvivalAnalysis(
                loss="coxph",
                random_state=CFG.random_state,
            ),
            param_grid={
                "n_estimators": [100, 220],
                "learning_rate": [0.03, 0.08],
                "max_depth": [1, 2],
                "subsample": [0.7, 1.0],
            },
        ),
    ]


class NestedSurvivalCV:
    def __init__(
        self,
        outer_splits=3,
        inner_splits=3,
        seed=42,
    ):
        self.outer_splits = outer_splits
        self.inner_splits = inner_splits
        self.seed = seed

    def run(self, X, y, specs):
        outer = StratifiedKFold(
            n_splits=self.outer_splits,
            shuffle=True,
            random_state=self.seed,
        )

        rows = []
        event = y["event"].astype(int)

        for fold, (tr, te) in enumerate(
            outer.split(X, event),
            start=1,
        ):
            Xtr, Xte = X.iloc[tr], X.iloc[te]
            ytr, yte = y[tr], y[te]

            inner = StratifiedKFold(
                n_splits=self.inner_splits,
                shuffle=True,
                random_state=self.seed + fold,
            )

            inner_cv = list(
                inner.split(
                    Xtr,
                    ytr["event"].astype(int),
                )
            )

            for spec in specs:
                search = GridSearchCV(
                    estimator=clone(spec.estimator),
                    param_grid=spec.param_grid,
                    scoring=None,
                    cv=inner_cv,
                    n_jobs=1,
                    refit=True,
                )

                search.fit(Xtr, ytr)

                risk = search.best_estimator_.predict(Xte)

                outer_c = concordance_index_censored(
                    yte["event"],
                    yte["time"],
                    risk,
                )[0]

                rows.append({
                    "outer_fold": fold,
                    "model": spec.name,
                    "outer_c_index": outer_c,
                    "inner_best_score": search.best_score_,
                    "best_params": search.best_params_,
                })

        return pd.DataFrame(rows)


nested_runner = NestedSurvivalCV(
    outer_splits=3,
    inner_splits=3,
    seed=CFG.random_state,
)

nested_results = nested_runner.run(
    X,
    y,
    build_nested_specs(CFG.fast_mode),
)

display(nested_results)

In [ ]:
nested_summary = (
    nested_results.groupby("model")
    .agg(
        mean_outer_c=("outer_c_index", "mean"),
        sd_outer_c=("outer_c_index", "std"),
        mean_inner_best=("inner_best_score", "mean"),
    )
    .reset_index()
)

display(nested_summary.round(4))

### Nested-CV interpretation

If the single-split score is much better than mean outer-CV performance, the single split may simply have been favorable.

This is especially important for small censored datasets where the **number of observed events**, not just sample size, determines effective statistical information.

# Part X — Competing risks with a real BMT dataset

## 16. Competing event types

The BMT dataset records:

- transplant-related mortality;
- relapse;
- right censoring.

If relapse is the target, transplant-related mortality is not ordinary independent censoring.

The correct absolute-risk quantity is the cumulative incidence function:

$$
F_k(t)
=
P(T\le t,J=k).
$$

It obeys

$$
F_k(t)
=
\int_0^t
S(u^-)\,dH_k(u).
$$

The subject must remain free of **all competing events** before experiencing cause $k$.

In [ ]:
X_bmt, y_bmt = load_bmt()

bmt = X_bmt.copy()
bmt["status"] = y_bmt["status"]
bmt["ftime"] = y_bmt["ftime"]

print("BMT shape:", bmt.shape)
display(bmt.head())

display(
    bmt["status"]
    .value_counts()
    .sort_index()
    .rename_axis("status")
    .to_frame("count")
)

In [ ]:
event_bmt = y_bmt["status"].astype(int)
time_bmt = y_bmt["ftime"].astype(float)

cif_times, cif_values = cumulative_incidence_competing_risks(
    event_bmt,
    time_bmt,
)

p = figure(
    width=820,
    height=430,
    title="BMT competing risks: cumulative incidence",
    x_axis_label="follow-up time",
    y_axis_label="cumulative incidence",
    y_range=(0, 1),
)

p.step(
    cif_times,
    cif_values[1],
    mode="after",
    line_width=3,
    legend_label="Transplant-related mortality",
)

p.step(
    cif_times,
    cif_values[2],
    mode="after",
    line_width=3,
    legend_label="Relapse",
)

p.legend.location = "top_left"
show(p)

## 17. Why naive $1-\hat S_{\text{KM}}$ is wrong here

A common mistake is to:

1. treat competing deaths as censored;
2. fit KM for relapse;
3. report

$$
1-\hat S_{\text{KM}}(t).
$$

That calculation behaves as though a person who died from another cause could remain eligible to relapse later.

We compare it with the proper CIF.

In [ ]:
relapse_event = event_bmt == 2

km_relapse_naive = KaplanMeierFitter().fit(
    durations=time_bmt,
    event_observed=relapse_event,
)

naive_failure = (
    1.0
    - km_relapse_naive
      .survival_function_
      .iloc[:, 0]
)

p = figure(
    width=820,
    height=430,
    title="Relapse: naive 1-KM versus competing-risk CIF",
    x_axis_label="time",
    y_axis_label="estimated probability",
    y_range=(0, 1),
)

p.step(
    naive_failure.index.to_numpy(dtype=float),
    naive_failure.to_numpy(dtype=float),
    mode="after",
    line_width=2,
    legend_label="Naive 1 - KM",
)

p.step(
    cif_times,
    cif_values[2],
    mode="after",
    line_width=3,
    legend_label="Competing-risk CIF",
)

p.legend.location = "top_left"
show(p)

## 18. Cause-specific prediction

For cause $k$:

$$
h_k(t\mid x)
=
h_{0k}(t)\exp(x^\top\beta_k).
$$

After fitting a model for each cause, combine hazard increments:

$$
\Delta F_k(t)
\approx
S(t^-)\Delta H_k(t).
$$

This section is pedagogical because the BMT dataset has only 35 subjects.

In [ ]:
def make_binary_survival_target(status, time, cause):
    return Surv.from_arrays(
        event=(status == cause),
        time=time.astype(float),
        name_event="event",
        name_time="time",
    )


X_bmt_num = X_bmt.astype(float)

cause_models = {}

for cause in [1, 2]:
    target = make_binary_survival_target(
        event_bmt,
        time_bmt,
        cause,
    )

    model = CoxPHSurvivalAnalysis(
        alpha=1e-4,
        ties="efron",
    )

    model.fit(X_bmt_num, target)
    cause_models[cause] = model


def predicted_cif_from_cause_specific_models(
    cause_models,
    X_profiles,
    times,
):
    n = len(X_profiles)
    causes = sorted(cause_models)

    H = {}

    for cause, model in cause_models.items():
        chf = model.predict_cumulative_hazard_function(
            X_profiles
        )

        H[cause] = np.row_stack([
            [float(fn(t)) for t in times]
            for fn in chf
        ])

    cif = {
        cause: np.zeros((n, len(times)))
        for cause in causes
    }

    cif_prev = {
        cause: np.zeros(n)
        for cause in causes
    }

    overall_H_prev = np.zeros(n)

    for j in range(len(times)):
        total_H_now = np.zeros(n)
        dH = {}

        for cause in causes:
            H_now = H[cause][:, j]
            H_before = (
                H[cause][:, j - 1]
                if j > 0
                else np.zeros(n)
            )

            dH[cause] = np.maximum(
                H_now - H_before,
                0.0,
            )

            total_H_now += H_now

        S_before = np.exp(-overall_H_prev)

        for cause in causes:
            cif_prev[cause] = (
                cif_prev[cause]
                + S_before * dH[cause]
            )
            cif[cause][:, j] = cif_prev[cause]

        overall_H_prev = total_H_now

    return cif


bmt_profiles = pd.DataFrame(
    {"dis": [0.0, 1.0]},
    index=["ALL", "AML"],
)

bmt_times = np.linspace(
    1,
    max(time_bmt) - 1e-6,
    100,
)

pred_cif = predicted_cif_from_cause_specific_models(
    cause_models,
    bmt_profiles,
    bmt_times,
)

p = figure(
    width=830,
    height=430,
    title="Cause-specific Cox prediction: relapse CIF",
    x_axis_label="time",
    y_axis_label="predicted relapse cumulative incidence",
    y_range=(0, 1),
)

for i, label in enumerate(bmt_profiles.index):
    p.line(
        bmt_times,
        pred_cif[2][i],
        line_width=3,
        legend_label=label,
    )

p.legend.location = "top_left"
show(p)

### Cause-specific hazard versus CIF

A cause-specific hazard ratio answers:

> Among subjects currently free of all terminal events, how does a covariate alter instantaneous cause-$k$ risk?

The CIF answers:

> What is the absolute probability that cause $k$ occurs by time $t$ before any competing cause?

These are different estimands.

# Part XI — Recurrent events

## 19. Why recurrent-event analysis differs

Many processes repeat:

- hospital admissions;
- infections;
- machine failures after repair;
- migraines;
- fraud incidents.

The recurrent dataset has multiple soreness episodes per person.

Each record represents an at-risk interval

$$
(\text{TIME0},\text{TIME1}].
$$

In [ ]:
recur = load_recur().copy()

print("Recurrent-event data shape:", recur.shape)
display(recur.head(10))

recur_summary = (
    recur.groupby("ID")
    .agg(
        age=("AGE", "first"),
        treatment=("TREAT", "first"),
        observed_events=("CENSOR", "sum"),
        followup=("TIME1", "max"),
        intervals=("EVENT", "count"),
    )
    .reset_index()
)

display(recur_summary.head())
print("Subjects:", recur_summary["ID"].nunique())
print("Observed recurrent events:", int(recur["CENSOR"].sum()))

## 20. Andersen–Gill counting-process model

Let

$$
N_i(t)
$$

count the events subject $i$ has experienced.

Let

$$
Y_i(t)
$$

indicate whether the subject is currently at risk.

A proportional-intensity model is

$$
\lambda_i(t)
=
Y_i(t)\lambda_0(t)\exp(x_i^\top\beta).
$$

The Cox risk-set machinery survives, but the same subject can re-enter the risk process after an event.

Repeated observations from one subject are correlated, so robust or random-effect inference becomes important.

In [ ]:
ag_data = recur[
    ["ID", "TIME0", "TIME1", "CENSOR", "AGE", "TREAT"]
].copy()

ag_model = CoxTimeVaryingFitter(
    penalizer=1e-4
)

try:
    ag_model.fit(
        ag_data,
        id_col="ID",
        start_col="TIME0",
        stop_col="TIME1",
        event_col="CENSOR",
        robust=True,
        show_progress=False,
    )
except Exception:
    ag_model.fit(
        ag_data,
        id_col="ID",
        start_col="TIME0",
        stop_col="TIME1",
        event_col="CENSOR",
        robust=False,
        show_progress=False,
    )

display(ag_model.summary)

In [ ]:
event_counts = (
    recur_summary["observed_events"]
    .value_counts()
    .sort_index()
)

p = figure(
    width=750,
    height=400,
    title="Distribution of observed recurrent-event counts",
    x_axis_label="number of observed episodes",
    y_axis_label="number of subjects",
)

p.vbar(
    x=event_counts.index.astype(float),
    top=event_counts.values,
    width=0.75,
)

show(p)

# Part XII — Frailty / random effects

## 21. Shared latent risk

Suppose two subjects have the same measured covariates but different unobserved susceptibility.

Introduce frailty

$$
Z_i>0.
$$

Then

$$
\lambda_i(t\mid Z_i)
=
Z_i\lambda_0(t)\exp(x_i^\top\beta).
$$

For Gamma frailty:

$$
Z_i
\sim
\operatorname{Gamma}
\left(
\frac1\theta,\theta
\right),
$$

so

$$
E[Z_i]=1
$$

and

$$
\operatorname{Var}(Z_i)=\theta.
$$

Large $\theta$ means substantial unobserved heterogeneity.

## 22. Frailty as recurrent-count overdispersion

Aggregate recurrent events by subject and suppose

$$
N_i\mid Z_i
\sim
\operatorname{Poisson}(Z_i\mu_i),
$$

where

$$
\mu_i
=
T_i\lambda_0\exp(x_i^\top\beta).
$$

Integrating over Gamma frailty yields a Negative-Binomial-like marginal model with

$$
E[N_i]=\mu_i
$$

and

$$
\operatorname{Var}(N_i)
=
\mu_i+\theta\mu_i^2.
$$

Thus $\theta$ directly measures excess between-subject heterogeneity.

In [ ]:
frailty_df = recur_summary.copy()

frailty_df["age_centered"] = (
    frailty_df["age"]
    - frailty_df["age"].mean()
) / frailty_df["age"].std(ddof=0)


def gamma_frailty_count_nll(params, data):
    """
    Marginal Gamma-Poisson negative log-likelihood.

    params:
      log_lambda0,
      beta_age,
      beta_treat,
      log_theta
    """
    log_lambda0, beta_age, beta_treat, log_theta = params

    theta = np.exp(log_theta)
    r = 1.0 / theta

    eta = (
        log_lambda0
        + beta_age
          * data["age_centered"].to_numpy()
        + beta_treat
          * data["treatment"].to_numpy()
    )

    exposure = np.clip(
        data["followup"].to_numpy(dtype=float),
        1e-6,
        None,
    )

    mu = exposure * np.exp(
        np.clip(eta, -30, 30)
    )

    n = data["observed_events"].to_numpy(
        dtype=float
    )

    ll = (
        gammaln(n + r)
        - gammaln(r)
        - gammaln(n + 1)
        + r * np.log(r / (r + mu))
        + n * np.log(mu / (r + mu))
    )

    return -np.sum(ll)


frailty_fit = minimize(
    gamma_frailty_count_nll,
    x0=np.array([
        -4.0,
        0.0,
        0.0,
        -1.0,
    ]),
    args=(frailty_df,),
    method="BFGS",
)

(
    log_lambda0,
    beta_age,
    beta_treat,
    log_theta,
) = frailty_fit.x

theta_hat = np.exp(log_theta)

frailty_result = pd.DataFrame({
    "parameter": [
        "baseline_rate_lambda0",
        "beta_age",
        "beta_treatment",
        "treatment_rate_ratio",
        "frailty_variance_theta",
    ],
    "estimate": [
        np.exp(log_lambda0),
        beta_age,
        beta_treat,
        np.exp(beta_treat),
        theta_hat,
    ],
})

display(frailty_result.round(4))
print("Optimization success:", frailty_fit.success)

### Interpreting $\hat\theta$

If

$$
\hat\theta\approx0,
$$

there is little residual between-subject heterogeneity after measured covariates.

If $\hat\theta$ is materially positive, the repeated-event process varies more across individuals than a simple homogeneous Poisson model can explain.

This is a transparent **frailty analogue**, not a substitute for a full semiparametric shared-frailty Cox model.

# Part XIII — Flexible parametric spline survival

## 23. Why splines?

A Weibull model imposes a rigid hazard shape.

Cox leaves the baseline hazard unspecified.

Flexible parametric survival modelling sits between them by representing a smooth baseline function with spline bases:

$$
g(t)
=
\sum_{k=1}^{K}
\gamma_k B_k(t).
$$

Benefits:

- smooth survival curves;
- flexible hazard shape;
- likelihood-based inference;
- possible extrapolation;
- explicit probability prediction.

In [ ]:
rossi_spline = rossi[
    [
        "week",
        "arrest",
        "fin",
        "age",
        "wexp",
        "mar",
        "paro",
        "prio",
    ]
].copy()


def fit_crc_spline(df, n_knots):
    regressors = {
        "beta_": (
            "fin + age + wexp + mar + paro + prio"
        )
    }

    for k in range(n_knots):
        regressors[f"gamma{k}_"] = "1"

    model = CRCSplineFitter(
        n_baseline_knots=n_knots
    )

    model.fit(
        df,
        duration_col="week",
        event_col="arrest",
        regressors=regressors,
    )

    return model


spline_models = {}
spline_rows = []

for knots in [2, 3, 4]:
    try:
        model = fit_crc_spline(
            rossi_spline,
            knots,
        )

        spline_models[knots] = model

        spline_rows.append({
            "baseline_knots": knots,
            "AIC": model.AIC_,
            "log_likelihood": model.log_likelihood_,
        })

    except Exception as exc:
        spline_rows.append({
            "baseline_knots": knots,
            "AIC": np.nan,
            "log_likelihood": np.nan,
            "note": str(exc)[:120],
        })


spline_comparison = pd.DataFrame(
    spline_rows
)

display(spline_comparison.round(3))

### Knot-count trade-off

As spline complexity increases,

$$
K\uparrow
\Rightarrow
\text{bias}\downarrow,
\quad
\text{variance}\uparrow.
$$

AIC balances likelihood fit against parameter count:

$$
AIC
=
-2\ell(\hat\theta)+2k.
$$

Among comparable likelihood-based models fitted to the same data, lower AIC is preferred.

In [ ]:
valid_splines = {
    k: m
    for k, m in spline_models.items()
    if hasattr(m, "predict_survival_function")
}

if valid_splines:
    spline_profile = pd.DataFrame([{
        col: float(rossi_spline[col].median())
        for col in [
            "fin",
            "age",
            "wexp",
            "mar",
            "paro",
            "prio",
        ]
    }])

    for c in ["fin", "wexp", "mar", "paro"]:
        spline_profile[c] = (
            spline_profile[c]
            .round()
            .clip(0, 1)
        )

    grid = np.linspace(1, 52, 100)

    p = figure(
        width=820,
        height=420,
        title="Flexible spline survival curves: knot sensitivity",
        x_axis_label="week",
        y_axis_label="survival probability",
        y_range=(0, 1),
    )

    for knots, model in valid_splines.items():
        sf = model.predict_survival_function(
            spline_profile,
            times=grid,
        )

        p.line(
            sf.index.to_numpy(dtype=float),
            sf.iloc[:, 0].to_numpy(dtype=float),
            line_width=3,
            legend_label=f"{knots} baseline knots",
        )

    p.legend.location = "bottom_left"
    show(p)

# Part XIV — Complexity and engineering trade-offs

## 24. High-level complexity intuition

### Cox PH

With efficient risk-set aggregation, sorting and cumulative sums avoid explicitly recomputing all risk sets.

Dense Newton updates still require matrix operations whose solve can scale roughly as

$$
O(p^3)
$$

per iteration.

### Coxnet

Coordinate descent exploits sparsity and warm starts along a regularization path.

It is especially useful when

$$
p\gg n.
$$

### Random Survival Forest

At a high level, work grows with:

$$
B\times
\text{tree-building cost}.
$$

Survival split evaluation is more expensive than ordinary impurity evaluation.

### Gradient boosting

Cost is approximately proportional to:

$$
M\times
\text{cost of one shallow tree}.
$$

### DeepSurv

Each epoch includes:

- neural forward pass;
- risk-set partial-likelihood evaluation;
- backpropagation.

The most flexible model is not necessarily the most efficient or statistically appropriate one.

# Part XV — Model-selection guide

| Situation | Strong starting point |
|---|---|
| Small tabular dataset, interpretation important | Cox PH |
| Many features / feature selection | Coxnet |
| Nonlinearities and interactions | RSF |
| Additive nonlinear risk structure | Survival gradient boosting |
| Large data / learned representations | DeepSurv-style Cox network |
| Multiple mutually exclusive terminal causes | Competing-risk model |
| Multiple events per subject | Recurrent-event model |
| Latent cluster/subject heterogeneity | Frailty/random effects |
| Smooth baseline + extrapolation | Flexible spline survival |

A mature workflow is:

$$
\boxed{
\text{estimand}
\rightarrow
\text{event structure}
\rightarrow
\text{assumptions}
\rightarrow
\text{model}
\rightarrow
\text{censoring-aware evaluation}
}
$$

# Part XVI — Common mistakes

### 1. Converting everything to binary classification

This throws away event timing and often mishandles subjects censored before the chosen horizon.

### 2. Reporting only C-index

Ranking quality does not imply calibrated probabilities.

### 3. Tuning on the test set

This produces optimistic performance.

### 4. Ignoring event balance across CV folds

Effective information is driven heavily by observed events.

### 5. Reading hazard ratio as probability ratio

$$
HR=0.7
$$

does not mean a 30% larger survival probability.

### 6. Treating competing events as ordinary censoring

A terminal competing event removes the possibility of the target event.

### 7. Ignoring within-subject dependence for recurrent events

Repeated episodes from one subject are correlated.

### 8. Assuming neural models automatically beat Cox

Model capacity must be supported by data capacity.

# Part XVII — Exercises

## Exercise 1 — RSF bias/variance

Vary:

- `min_samples_split`;
- `min_samples_leaf`;
- `max_features`;
- `n_estimators`.

Record:

- Harrell C;
- Uno C;
- IBS.

Find a case where C-index improves but IBS worsens.

---

## Exercise 2 — Gradient-boosting losses

Try:

```python
loss="coxph"
loss="ipcwls"
loss="squared"
```

Explain how the prediction semantics change.

---

## Exercise 3 — Coxnet sparsity

Try:

```python
l1_ratio = 1.0
l1_ratio = 0.9
l1_ratio = 0.5
l1_ratio = 0.1
```

Compare coefficient paths.

---

## Exercise 4 — DeepSurv architecture

Compare:

```python
hidden=(16,)
hidden=(32,16)
hidden=(64,32,16)
```

Measure:

- train loss;
- validation loss;
- C-index;
- IBS.

---

## Exercise 5 — Calibration horizons

Repeat at

$$
t=13,\;26,\;39.
$$

Can a model be calibrated early but not late?

---

## Exercise 6 — Nested CV

Increase the number of outer folds.

Compare mean and standard deviation of outer C-index.

---

## Exercise 7 — Competing risks

Estimate CIF separately for ALL and AML.

Explain why

$$
F_{\text{relapse}}(t)
+
F_{\text{TRM}}(t)
\le1.
$$

---

## Exercise 8 — Recurrent events

Add episode number to the recurrent-event model and investigate event-order effects.

This leads toward Prentice–Williams–Peterson-style models.

---

## Exercise 9 — Frailty

Investigate whether

$$
\hat\theta
$$

is meaningfully above zero.

---

## Exercise 10 — Flexible splines

Increase spline knots and compare:

- AIC;
- likelihood;
- survival-curve stability.

# Part XVIII — Final conceptual map

```text
Censoring + risk sets
        |
        +-------------------------------+
        |                               |
   Cox partial likelihood          IPCW / censoring KM
        |                               |
   +----+---------+                evaluation
   |              |                    |
Coxnet      DeepSurv / GB         Uno C / AUC / IBS

Kaplan-Meier / Nelson-Aalen
        |
terminal-node survival
        |
Random Survival Forest

Multiple terminal causes
        |
cause-specific hazards
        |
Cumulative incidence

Repeated events
        |
counting-process Cox
        |
frailty / random effects

Rigid parametric baseline
        |
spline basis expansion
        |
flexible parametric survival
```

The central lesson is:

$$
\boxed{
\text{Modern survival ML}
=
\text{classical censoring theory}
+
\text{more flexible function approximation}
}
$$

The classical ST5212 material therefore remains foundational.

# References / API cross-check

The notebook is designed around current APIs for:

- `RandomSurvivalForest`;
- `GradientBoostingSurvivalAnalysis`;
- `CoxnetSurvivalAnalysis`;
- censoring-aware Brier score / IBS;
- cumulative dynamic AUC;
- `CensoringDistributionEstimator`;
- competing-risk cumulative incidence;
- `load_bmt`;
- `load_recur`;
- `CoxTimeVaryingFitter`;
- `CRCSplineFitter`.

Documentation:

- https://scikit-survival.readthedocs.io/
- https://scikit-survival.readthedocs.io/en/stable/user_guide/coxnet.html
- https://scikit-survival.readthedocs.io/en/stable/user_guide/competing-risks.html
- https://lifelines.readthedocs.io/

Suggested theory references:

- Kalbfleisch & Prentice — *The Statistical Analysis of Failure Time Data*;
- Klein & Moeschberger — *Survival Analysis*;
- Therneau & Grambsch — *Modeling Survival Data: Extending the Cox Model*;
- Ishwaran et al. — Random Survival Forests;
- Simon et al. — Cox regularization paths;
- Katzman et al. — DeepSurv.